In [ ]:
import sys, os

#change the path to the directory that ctmc.py is located in on your system
sys.path.append(os.path.expanduser('~/source/discrete_states/'))

In [ ]:
from ctmc import ContinuousTimeMarkovChain as MC
from ctmc import normal_generator, gamma_generator, uniform_generator, cyclic_generator, detailed_balance_generator, arrhenius_pump_generator

import numpy as np
import matplotlib.pyplot as plt 

In [ ]:
#you can start by making a bespoke rate equation if you want. 
R = abs(np.empty((3,3)))
# we can set the rates by hand, to be DB
E_0 = 1
E_1 = 2
E_2 = 3
R[0,1] = np.exp(-E_1+E_0) 
R[1,0] = 1/R[0,1]
R[0,2] = np.exp(-E_2+E_0) 
R[2,0] = 1/R[0,2]
R[1,2] = np.exp(-E_2+E_1)
R[2,1] = 1/R[1,2]

# and for fun, let's add something non DB
R[0,2] *= np.exp(4)

#the rate matrix looks like
print(R)
machine = MC(R)

In [ ]:

print("NESS distribution")
print(machine.get_ness())
print("MEPS distribution")
print(machine.get_meps())

print('meps: ', machine.get_epr(machine.meps), 'ness: ', machine.get_epr(machine.ness))

In [ ]:
machine = MC(R)
print("Rate matrix")
print(machine.R)

print("Reverse rate matrix")
print(machine.rev_R)

In [ ]:
# R = rev_R because this is set to True:
print('time_even states: ', machine.time_even_states)

### CURRENT METHOD TO GET TIME ASSYMETRY

In [ ]:
# 1 set the time symmetry to False
machine.time_even_states = False

### STEP 2 has multiple options

- manually create an involution
- get a random one

In [ ]:
# set a manual involution by indices

# can skip this step if you want a random involution instead
# this involution will swap state 0 and 2, and leave state 1 alone
machine.involution_indices = np.array([2,1,0])
print(machine.involution_indices)



In [ ]:
# now, we can get the reversal matrix
# this is usually an internal tool, but could be useful to access for bug fixing
machine.get_reversal_matrix()

In [ ]:
# we sohuld also be able to to zero probability transitions that are aware of time symmetry

machine.R[0,1] = 0
machine.set_rate_matrix(machine.R)
print(machine.R)

print(machine.rev_R.T)

In [ ]:

machine.get_ness()
machine.get_meps()

print('meps epr: ', machine.get_epr(machine.meps), 'ness epr: ', machine.get_epr(machine.ness), 'dkl: ', machine.dkl(machine.meps, machine.ness))


#### Here is how you would make your own involution randomly for larger machine w. zero transition rates

In [ ]:

machines= MC(generator=cyclic_generator, S=9, N=3, max_reverse_rate=.5, max_jump=2, mu=.5)

In [ ]:
# Here we have a kind of finite range cycle, with the first and last states connected
# here, there are a lot of zero transiton rates, to check it is working well.


In [ ]:
# Visualize the rate matrix in log scale
fig, ax = plt.subplots(figsize=(8, 7))

# Get the rate matrix and mask zero values for log scale
R_plot = machines.R[0].copy()
# Replace zeros with NaN for better visualization (they'll show as white/gray)
R_plot_masked = np.where(R_plot > 0, R_plot, np.nan)

# Create log-scale plot
im = ax.imshow(np.log10(R_plot_masked), cmap='viridis', aspect='auto')


# Add colorbar
cbar = plt.colorbar(im, ax=ax)
cbar.set_label('log10(Rate)', rotation=270, labelpad=20)

# Add grid and labels
ax.set_xticks(np.arange(machines.S))
ax.set_yticks(np.arange(machines.S))
ax.set_xlabel('To State')
ax.set_ylabel('From State')
ax.set_title('Rate Matrix R[0] (log scale)')

# Add text annotations showing actual values
for i in range(machines.S):
    for j in range(machines.S):
        if R_plot[i, j] > 0:
            text = ax.text(j, i, f'{R_plot[i, j]:.2e}',
                          ha="center", va="center", color="w", fontsize=8)
        elif R_plot[i, j] == 0 and i != j:
            # Show zeros as "0" in gray
            text = ax.text(j, i, '0',
                          ha="center", va="center", color="gray", fontsize=8)
        elif R_plot[i, j] < 0 and i==j:
            text = ax.text(j, i, f'{R_plot[i, j]:.2e}',
                          ha="center", va="center", color="red", fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
# you might have to run this cell 3-5 times for meps to converge
print('ness epr: ' , machines.get_epr(machines.get_ness() ))
print('meps epr: ' , machines.get_epr(machines.get_meps() ))

In [ ]:
# ok now we will try to do it with time ayssymetry

In [ ]:
machines.time_even_states = False

# if you want a random involution:
machines.get_reversal_matrix()

# if you want to make your own
#machines.involution_indices = np.array([8, 1, 3, 2, 4, 7, 6, 5, 0])

In [ ]:
machines.set_rate_matrix(machines.R)

In [ ]:
print('ness epr: ' , machines.get_epr(machines.get_ness() ))
print('meps epr: ' , machines.get_epr(machines.get_meps() ))